In [1]:
import pandas as pd
import random
from datetime import date, timedelta

# ── Seed for reproducibility ──────────────────────────────────────────────────
random.seed(42)

# ═══════════════════════════════════════════════════════════════════════════════
# 1. dim_channels
# ═══════════════════════════════════════════════════════════════════════════════
dim_channels = pd.DataFrame({
    "channel_id":              ["CH_OTA_BKG", "CH_OTA_EXP", "CH_DIRECT", "CH_WHOLE"],
    "channel_name":            ["Booking.com", "Expedia", "Direct Website", "Wholesale Partner"],
    "channel_type":            ["OTA", "OTA", "Direct", "Wholesale"],
    "commission_model":        ["Percentage", "Percentage", "Marketing Cost", "Net Rate"],
    "default_commission_rate": [0.18, 0.15, 0.00, 0.00],
})
dim_channels.to_csv("dim_channels.csv", index=False)
print("✓ dim_channels.csv saved")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. dim_rate_codes
# ═══════════════════════════════════════════════════════════════════════════════
dim_rate_codes = pd.DataFrame({
    "rate_code_id":    ["RACK", "PROMO", "CORP", "NET"],
    "rate_name":       ["Rack Rate", "Promotional Rate", "Corporate Rate", "Net Rate"],
    "is_commissionable": [True, True, True, False],
})
dim_rate_codes.to_csv("dim_rate_codes.csv", index=False)
print("✓ dim_rate_codes.csv saved")

# ── Lookup maps used by fact tables ───────────────────────────────────────────
commission_rate_map    = dict(zip(dim_channels["channel_id"],
                                  dim_channels["default_commission_rate"]))
is_commissionable_map  = dict(zip(dim_rate_codes["rate_code_id"],
                                  dim_rate_codes["is_commissionable"]))

channel_ids   = dim_channels["channel_id"].tolist()
rate_code_ids = dim_rate_codes["rate_code_id"].tolist()

# ── Date helpers ──────────────────────────────────────────────────────────────
START_DATE = date(2025, 1, 1)
END_DATE   = date(2026, 12, 31)
DATE_RANGE = (END_DATE - START_DATE).days

def rand_date(start=START_DATE, days=DATE_RANGE):
    return start + timedelta(days=random.randint(0, days))

# ═══════════════════════════════════════════════════════════════════════════════
# 3. fact_bookings  (5 000 rows)
# ═══════════════════════════════════════════════════════════════════════════════
N_BOOKINGS = 5_000
rows = []

for i in range(N_BOOKINGS):
    booking_id   = f"BK_{i:05d}"
    booking_date = rand_date()
    # check-in must be AFTER booking date; offset 1-90 days ahead
    check_in_date = booking_date + timedelta(days=random.randint(1, 90))

    channel_id   = random.choice(channel_ids)
    rate_code_id = random.choice(rate_code_ids)
    rooms_sold   = random.randint(1, 3)

    # Gross revenue: rooms × nightly rate (80–500 USD, 1–5 nights implied in rate)
    gross_room_revenue = round(random.uniform(80, 500) * rooms_sold, 2)

    # Commission logic
    if is_commissionable_map[rate_code_id]:
        commission_amount = round(gross_room_revenue * commission_rate_map[channel_id], 2)
    else:
        commission_amount = 0.00

    net_room_revenue = round(gross_room_revenue - commission_amount, 2)
    status = random.choice(["Confirmed", "Cancelled", "Checked-Out"])

    rows.append({
        "booking_id":         booking_id,
        "booking_date":       booking_date,
        "check_in_date":      check_in_date,
        "channel_id":         channel_id,
        "rate_code_id":       rate_code_id,
        "rooms_sold":         rooms_sold,
        "gross_room_revenue": gross_room_revenue,
        "commission_amount":  commission_amount,
        "net_room_revenue":   net_room_revenue,
        "status":             status,
    })

fact_bookings = pd.DataFrame(rows)
fact_bookings.to_csv("fact_bookings.csv", index=False)
print("✓ fact_bookings.csv saved")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. fact_marketing_spend  (120 rows)
# ═══════════════════════════════════════════════════════════════════════════════
N_SPEND = 120
spend_rows = []

for i in range(N_SPEND):
    spend_rows.append({
        "spend_id":    f"SP_{i:03d}",
        "spend_date":  rand_date(),
        "channel_id":  "CH_DIRECT",                          # always Direct
        "platform":    random.choice(["Google Ads", "Facebook"]),
        "cost_amount": random.randint(200, 5_000),           # USD ad spend
    })

fact_marketing_spend = pd.DataFrame(spend_rows)
fact_marketing_spend.to_csv("fact_marketing_spend.csv", index=False)
print("✓ fact_marketing_spend.csv saved")

# ── Quick sanity-check printout ───────────────────────────────────────────────
print("\n── dim_channels ──────────────────────────────────────────────────────")
print(dim_channels.to_string(index=False))

print("\n── dim_rate_codes ────────────────────────────────────────────────────")
print(dim_rate_codes.to_string(index=False))

print("\n── fact_bookings (first 5 rows) ──────────────────────────────────────")
print(fact_bookings.head().to_string(index=False))

print("\n── fact_marketing_spend (first 5 rows) ───────────────────────────────")
print(fact_marketing_spend.head().to_string(index=False))

print(f"\nRow counts → fact_bookings: {len(fact_bookings):,} | "
      f"fact_marketing_spend: {len(fact_marketing_spend)}")

✓ dim_channels.csv saved
✓ dim_rate_codes.csv saved
✓ fact_bookings.csv saved
✓ fact_marketing_spend.csv saved

── dim_channels ──────────────────────────────────────────────────────
channel_id      channel_name channel_type commission_model  default_commission_rate
CH_OTA_BKG       Booking.com          OTA       Percentage                     0.18
CH_OTA_EXP           Expedia          OTA       Percentage                     0.15
 CH_DIRECT    Direct Website       Direct   Marketing Cost                     0.00
  CH_WHOLE Wholesale Partner    Wholesale         Net Rate                     0.00

── dim_rate_codes ────────────────────────────────────────────────────
rate_code_id        rate_name  is_commissionable
        RACK        Rack Rate               True
       PROMO Promotional Rate               True
        CORP   Corporate Rate               True
         NET         Net Rate              False

── fact_bookings (first 5 rows) ──────────────────────────────────────
booking_